In [1]:
from pyspark.sql.functions import col, trim, initcap, when, lit, create_map, current_timestamp
from itertools import chain

bronze = spark.table("bronze_revenue_transactions")

region_map = {"NE": "Northeast", "SE": "Southeast", "SW": "Southwest"}
mapping_expr = create_map([lit(x) for x in chain(*region_map.items())])

silver = (bronze
    .withColumn("region", initcap(trim(col("region"))))
    .withColumn("region", when(col("region").isin(list(region_map.keys())), mapping_expr[col("region")]).otherwise(col("region")))
    .dropDuplicates(["order_id"]))

cust_lookup = (silver.filter(col("customer_name").isNotNull())
               .select("customer_id", "customer_name").dropDuplicates(["customer_id"]))
silver = silver.drop("customer_name").join(cust_lookup, on="customer_id", how="left")

silver = silver.withColumn("_processed_at", current_timestamp())
silver.write.format("delta").mode("overwrite").saveAsTable("silver_revenue_transactions")

StatementMeta(, c639cfa0-0b61-47c6-87ff-e28d8cb667df, 3, Finished, Available, Finished, False)